# Analisis Spasial & Klastering Pasar Kerja Pulau Jawa
**Proyek Analisis Klastering Lowongan Kerja (Jobstreet) & Sosio-Ekonomi (BPS) dengan DBSCAN**

Notebook ini merangkum seluruh alur pengolahan data dari awal hingga akhir (end-to-end), meliputi:
1. **Tahap 1: Data Acquisition (Scraping)**: Penjelasan mengenai mekanisme akuisisi lowongan kerja Jobstreet.
2. **Tahap 2: Socio-Economic Data Cleaning**: Konsolidasi data ketenagakerjaan dari Badan Pusat Statistik (BPS).
3. **Tahap 3: Geocoding**: Pencarian koordinat wilayah 119 Kabupaten/Kota di Jawa menggunakan Nominatim (OSM).
4. **Tahap 4: Data Fusion & Fuzzy Matching**: Integrasi lowongan dengan wilayah BPS menggunakan pencocokan teks fuzzy.
5. **Tahap 5: Opportunity Index Calculation**: Penghitungan rasio penyerapan tenaga kerja per wilayah.
6. **Tahap 6: Spatial Clustering (DBSCAN)**: Pemodelan aglomerasi hub ekonomi spasial berbasis algoritma kepadatan spasial.
7. **Tahap 7: Visualisasi Eksploratif**: Pemetaan spasial hasil klastering menggunakan Matplotlib & Seaborn.

---

In [ ]:
# Install library pendukung jika belum terinstall
# !pip install pandas numpy scikit-learn rapidfuzz geopy matplotlib seaborn openpyxl statsmodels plotly scipy

## Tahap 1: Akuisisi Data Lowongan Kerja (Jobstreet API)
Pada tahap ini, data lowongan kerja diakuisisi secara langsung dari API pencarian Jobstreet menggunakan reverse engineering GraphQL endpoint.
Kode di bawah ini disiapkan untuk menunjukkan cara akuisisi data. Jika file `data/jobstreet_results.csv` sudah ada di sistem Anda, sel ini dapat dilewati.

In [ ]:
import pandas as pd
import os

CSV_PATH = 'data/jobstreet_results.csv'

if os.path.exists(CSV_PATH):
    print(f"File data lowongan ditemukan: {CSV_PATH}")
    df_js_preview = pd.read_csv(CSV_PATH)
    print(f"Jumlah Lowongan Terdaftar: {len(df_js_preview)} lowongan.")
    display(df_js_preview.head(3))
else:
    print(f"[PERINGATAN] File {CSV_PATH} tidak ditemukan.")
    print("Mekanisme pengambilan data secara live membutuhkan cookies & bearer token aktif dari platform Jobstreet.")

## Tahap 2: Konsolidasi Data Sosio-Ekonomi BPS
Tahap ini menggabungkan 6 file CSV data ketenagakerjaan dari tingkat provinsi (BPS 2025) yang tersimpan di dalam folder `data-bps/` menjadi satu dataset tunggal. 
Proses pembersihan mencakup penghapusan notasi angka awalan wilayah dan perbaikan data format Excel yang mengalami korupsi nilai (seperti pemisahan desimal ribuan).

In [ ]:
import pandas as pd
import glob
import os

"""
TAHAP 2.1: KONSOLIDASI DATA BPS
Penulis: Antigravity AI (Falah's Thesis Assistant)
Deskripsi: Script ini menggabungkan 6 file CSV sosio-ekonomi dari tingkat provinsi 
           menjadi satu dataset master Kabupaten/Kota se-Pulau Jawa.
"""

def main():
    print("Memulai Konsolidasi Data BPS...")
    
    # Path folder data mentah
    data_path = 'data-bps/*.csv'
    files = glob.glob(data_path)
    
    all_data = []
    
    for f in files:
        # Mengambil nama provinsi dari nama file
        provinsi = os.path.basename(f).split('di Provinsi ')[-1].split(',')[0].strip()
        print(f"Memproses Provinsi: {provinsi}")
        
        # Membaca data dengan encoding yang sesuai
        df = pd.read_csv(f)
        df['Provinsi'] = provinsi
        all_data.append(df)
    
    # Menggabungkan semua data
    master_df = pd.concat(all_data, ignore_index=True)
    
    # Pembersihan Nama Kabupaten/Kota (Menghilangkan angka awalan jika ada)
    # Beberapa data BPS memiliki format [3171] Kota Jakarta Pusat
    def clean_name(name):
        if not isinstance(name, str): return name
        import re
        return re.sub(r'\[.*?\]\s*', '', name).strip()

    master_df['Kabupaten/Kota'] = master_df['Kabupaten/Kota'].apply(clean_name)
    
    # Pembersihan kolom sosio-ekonomi (numerik)
    def clean_bps_val(val):
        import re
        if pd.isna(val):
            return 0
        val_str = str(val).strip()
        
        # Hapus catatan kaki seperti " (a)"
        val_str = re.sub(r'\s*\(.*?\)', '', val_str)
        if not val_str or val_str.lower() == 'nan':
            return 0
            
        # Periksa format tanggal Excel (seperti 1/7/90 atau 1/20/62)
        if '/' in val_str:
            parts = val_str.split('/')
            if len(parts) == 3:
                millions = parts[0]
                thousands = parts[1].zfill(3)
                units = parts[2].zfill(3)
                return float(f"{millions}{thousands}{units}")
                
        # Hapus tanda koma
        val_str = val_str.replace(',', '')
        
        # Periksa separator titik
        if '.' in val_str:
            parts = val_str.split('.')
            if len(parts) > 2:
                # Titik ganda seperti 1.007.090
                return float("".join(parts))
            elif len(parts) == 2:
                # Titik tunggal seperti 569.654 atau 385.8
                if len(parts[1]) == 3:
                    return float("".join(parts))
                elif len(parts[1]) < 3:
                    # Desimal ribuan seperti 385.8 -> 385800
                    padded_right = parts[1].ljust(3, '0')
                    return float(f"{parts[0]}{padded_right}")
                else:
                    return float("".join(parts))
        else:
            try:
                val_float = float(val_str)
                # Koreksi untuk nilai integer bulat kecil (seperti 133 untuk Kota Probolinggo)
                # yang dibulatkan oleh Excel dari ribuan murni (133.000 -> 133)
                if 0 < val_float < 5000:
                    return val_float * 1000
                return val_float
            except ValueError:
                return 0

    # Terapkan pembersihan numerik ke semua kolom kecuali nama wilayah dan provinsi
    for col in master_df.columns:
        if col not in ['Kabupaten/Kota', 'Provinsi']:
            master_df[col] = master_df[col].apply(clean_bps_val)
    
    # Simpan ke Master CSV (di folder data)
    os.makedirs('data', exist_ok=True)
    output_file = os.path.join('data', 'master_bps_socioeconomic.csv')
    master_df.to_csv(output_file, index=False)
    
    print(f"Berhasil! Master data BPS disimpan di: {output_file}")
    print(f"Total baris data: {len(master_df)}")

if __name__ == "__main__":
    main()


## Tahap 3: Geocoding Wilayah (Mencari Koordinat Centroid)
Tahap ini mengambil koordinat (Latitude & Longitude) pusat wilayah dari 119 Kabupaten/Kota di Pulau Jawa menggunakan OpenStreetMap (Nominatim).
Untuk mencegah beban berlebih (rate limiting) pada server OpenStreetMap, geocoder dilengkapi dengan interval waktu 1 detik. Jika file `data/java_regency_coordinates.csv` sudah ada, tahap pencarian ini bisa dilewati untuk mempercepat proses.

In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

"""
TAHAP 2.2: GEOCODING KABUPATEN/KOTA
Penulis: Antigravity AI (Falah's Thesis Assistant)
Deskripsi: Script ini mengambil koordinat (Latitude & Longitude) pusat wilayah 
           untuk 119 Kabupaten/Kota di Pulau Jawa menggunakan OpenStreetMap (Nominatim).
"""

def main():
    print("Memulai Proses Geocoding...")
    
    # 1. Membaca daftar kota yang sudah disiapkan dari BPS
    try:
        with open('java_cities_list.txt', 'r') as f:
            cities = [line.strip() for line in f if line.strip()]
    except FileNotFoundError:
        print("Error: file java_cities_list.txt tidak ditemukan.")
        return

    # 2. Inisialisasi Geolocator
    geolocator = Nominatim(user_agent="skripsi_clustering_java")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

    results = []
    
    print(f"Mengambil koordinat untuk {len(cities)} wilayah...")
    for city in cities:
        # Override khusus untuk daerah dengan nama yang mirip di luar Jawa
        if city == 'Batang':
            search_query = "Kabupaten Batang, Jawa Tengah, Indonesia"
        elif city == 'Kota Banjar':
            search_query = "Kota Banjar, Jawa Barat, Indonesia"
        elif not city.startswith("Kota") and not city.startswith("Kepulauan"):
            search_query = f"Kabupaten {city}, Indonesia"
        else:
            search_query = f"{city}, Indonesia"
            
        try:
            location = geocode(search_query)
            
            # Fallback tanpa prefix jika tidak ditemukan
            if not location and "Kabupaten " in search_query:
                print(f"[RETRY] {city}: Mencoba kembali tanpa prefix...")
                fallback_query = f"{city}, Indonesia"
                location = geocode(fallback_query)
                
            if location:
                results.append({
                    "City_Name": city,
                    "Latitude": location.latitude,
                    "Longitude": location.longitude
                })
                print(f"[OK] {city}: {location.latitude}, {location.longitude}")
            else:
                print(f"[FAILED] {city}: Tidak ditemukan.")
        except Exception as e:
            print(f"[ERROR] {city}: {str(e)}")
            time.sleep(2) # Backoff jika ada error jaringan

    # 3. Simpan ke CSV di folder data
    import os
    os.makedirs('data', exist_ok=True)
    output_df = pd.DataFrame(results)
    output_file = os.path.join('data', 'java_regency_coordinates.csv')
    output_df.to_csv(output_file, index=False)
    
    print(f"\nProses Selesai! Koordinat disimpan di: {output_file}")

if __name__ == "__main__":
    main()


## Tahap 4: Integrasi Data Spasial & Sosio-Ekonomi (Data Fusion)
Pada tahap ini, data mentah lowongan kerja dari Jobstreet digabungkan dengan data koordinat wilayah dan data kependudukan BPS.
Karena format penamaan kota pada Jobstreet seringkali tidak standar (misal: "Cikarang" yang mengacu ke Kabupaten Bekasi, atau "Purwokerto" ke Banyumas), kita menggunakan pencocokan teks menggunakan algoritma **Fuzzy String Matching (RapidFuzz)** dengan tabel penyelamatan (rescue mapping) manual untuk menyelaraskan nama daerah dengan tingkat kecocokan minimum (threshold) 80%.

In [ ]:
import pandas as pd
from rapidfuzz import process, utils
import os

"""
TAHAP 3: INTEGRASI DATA (DATA FUSION)
Penulis: Antigravity AI (Falah's Thesis Assistant)
Deskripsi: Script ini menggabungkan data lowongan kerja (Jobstreet) dengan 
           koordinat wilayah dan data kependudukan (BPS) menggunakan teknik 
           Fuzzy String Matching untuk sinkronisasi nama wilayah.
"""

def main():
    print("Memulai Integrasi Data Spasial & Sosio-Ekonomi...")
    
    # 1. Memuat Dataset
    df_js = pd.read_csv('data/jobstreet_results.csv')
    df_coords = pd.read_csv('data/java_regency_coordinates.csv')
    df_bps = pd.read_csv('data/master_bps_socioeconomic.csv')

    # 2. Persiapan Daftar Wilayah (Master Regency)
    lookup_list = df_coords['City_Name'].tolist()
    
    # 3. Fungsi Pemetaan Fuzzy (Rescue Mapping)
    # Memetakan nama kecamatan/kawasan industri ke Kabupaten/Kota induknya
    cache = {
        "Bandung, Jawa Barat": ("Kota Bandung", 100),
        "Bandung": ("Kota Bandung", 100),
        "Kabupaten Bandung, Jawa Barat": ("Bandung", 100),
        "Bogor, Jawa Barat": ("Kota Bogor", 100),
        "Bogor": ("Kota Bogor", 100),
        "Kabupaten Bogor, Jawa Barat": ("Bogor", 100),
        "Bekasi, Jawa Barat": ("Kota Bekasi", 100),
        "Bekasi": ("Kota Bekasi", 100),
        "Kabupaten Bekasi, Jawa Barat": ("Bekasi", 100),
        "Tangerang, Banten": ("Kota Tangerang", 100),
        "Tangerang": ("Kota Tangerang", 100),
        "Semarang, Jawa Tengah": ("Kota Semarang", 100),
        "Semarang": ("Kota Semarang", 100),
        "Surabaya, Jawa Timur": ("Kota Surabaya", 100),
        "Surabaya": ("Kota Surabaya", 100),
        "Malang, Jawa Timur": ("Kota Malang", 100),
        "Malang": ("Kota Malang", 100),
        "Yogyakarta, DI Yogyakarta": ("Kota Yogyakarta", 100),
        "Yogyakarta": ("Kota Yogyakarta", 100),
        "Cirebon, Jawa Barat": ("Kota Cirebon", 100),
        "Cirebon": ("Kota Cirebon", 100),
        "Sukabumi, Jawa Barat": ("Kota Sukabumi", 100),
        "Sukabumi": ("Kota Sukabumi", 100),
        "Tegal, Jawa Tengah": ("Kota Tegal", 100),
        "Tegal": ("Kota Tegal", 100),
        "Magelang, Jawa Tengah": ("Kota Magelang", 100),
        "Magelang": ("Kota Magelang", 100),
        "Tasikmalaya, Jawa Barat": ("Kota Tasikmalaya", 100),
        "Tasikmalaya": ("Kota Tasikmalaya", 100),
        "Madiun, Jawa Timur": ("Kota Madiun", 100),
        "Madiun": ("Kota Madiun", 100),
        "Pasuruan, Jawa Timur": ("Kota Pasuruan", 100),
        "Pasuruan": ("Kota Pasuruan", 100),
        "Mojokerto, Jawa Timur": ("Kota Mojokerto", 100),
        "Mojokerto": ("Kota Mojokerto", 100),
        "Kediri, Jawa Timur": ("Kota Kediri", 100),
        "Kediri": ("Kota Kediri", 100),
        "Cikarang Pusat, Jawa Barat": ("Kota Bekasi", 100),
        "Cikarang, Jawa Barat": ("Kota Bekasi", 100),
        "Cikarang": ("Kota Bekasi", 100),
        "Kebayoran Lama, Jakarta Raya": ("Kota Jakarta Selatan", 100),
        "Kebayoran Baru, Jakarta Raya": ("Kota Jakarta Selatan", 100),
        "Kemayoran, Jakarta Raya": ("Kota Jakarta Pusat", 100),
        "Cikupa, Banten": ("Kota Tangerang", 100),
        "Ciawi, Jawa Barat": ("Kota Bogor", 100),
        "Serpong, Banten": ("Kota Tangerang Selatan", 100),
        "Bsd City, Banten": ("Kota Tangerang Selatan", 100),
        "Cileungsi, Jawa Barat": ("Kota Bogor", 100),
        "Kalideres, Jakarta Raya": ("Kota Jakarta Barat", 100),
        "Kelapa Gading, Jakarta Raya": ("Kota Jakarta Utara", 100),
        "Gunung Putri, Jawa Barat": ("Kota Bogor", 100),
        "Purwokerto": ("Banyumas", 100),
        "Padalarang, Jawa Barat": ("Bandung Barat", 100),
        "Pulo Gadung, Jakarta Raya": ("Kota Jakarta Timur", 100),
        "Waru, Jawa Timur": ("Sidoarjo", 100),
        "Cengkareng, Jakarta Raya": ("Kota Jakarta Barat", 100),
        "Balaraja, Banten": ("Tangerang", 100),
        "Penjaringan, Jakarta Raya": ("Kota Jakarta Utara", 100),
        "Tambun, Jawa Barat": ("Bekasi", 100),
        "Matraman, Jakarta Raya": ("Kota Jakarta Timur", 100),
        "Sunter, Jakarta Raya": ("Kota Jakarta Utara", 100),
        "Cikande, Banten": ("Serang", 100),
        "Pesanggrahan, Jakarta Raya": ("Kota Jakarta Selatan", 100),
        "Gunung Sindur, Jawa Barat": ("Bogor", 100),
        "Driyorejo, Jawa Timur": ("Gresik", 100),
        "Jetis, DI Yogyakarta": ("Kota Yogyakarta", 100),
        "Batujajar, Jawa Barat": ("Bandung Barat", 100),
        "Gedangan, Jawa Timur": ("Sidoarjo", 100),
        "Kebon Jeruk, Jakarta Raya": ("Kota Jakarta Barat", 100),
        "Setiabudi, Jakarta Raya": ("Kota Jakarta Selatan", 100),
    }

    def get_best_match(loc_string):
        if not isinstance(loc_string, str) or not loc_string.strip():
            return None, 0
        
        loc_string = loc_string.strip()
        if loc_string in cache:
            return cache[loc_string]
        
        # Ambil bagian utama nama lokasi (sebelum koma)
        primary = loc_string.split(',')[0].strip()
        
        # Gunakan fuzzy matching untuk mencari kecocokan terbaik
        match = process.extractOne(primary, lookup_list, processor=utils.default_process)
        if match:
            cache[loc_string] = (match[0], match[1])
            return match[0], match[1]
        return None, 0

    # 4. Menjalankan Pemetaan
    print(f"Memetakan {len(df_js)} data lowongan ke {len(lookup_list)} wilayah standar...")
    df_js[['matched_regency', 'match_score']] = df_js['location'].apply(
        lambda x: pd.Series(get_best_match(x))
    )

    # 5. Filter Data Valid (Threshold > 80)
    # Menghapus data yang tidak spesifik (misal: "Jawa Timur") atau tidak ada koordinatnya
    good_matches = df_js[df_js['match_score'] >= 80].copy()
    
    # 6. Penggabungan Koordinat
    final_df = pd.merge(
        good_matches, 
        df_coords, 
        left_on='matched_regency', 
        right_on='City_Name', 
        how='left'
    )
    
    # 7. Penggabungan Data BPS secara presisi
    # Tidak lagi menghapus awalan 'Kota'/'Kabupaten' agar tidak tertimpa/hilang
    final_df = pd.merge(
        final_df,
        df_bps,
        left_on='matched_regency',
        right_on='Kabupaten/Kota',
        how='left'
    )

    # 8. Seleksi Kolom Akhir
    cols_to_keep = [
        'id', 'title', 'company', 'location', 'matched_regency', 'match_score',
        'Latitude', 'Longitude', 'Provinsi',
        'Angkatan Kerja - Bekerja', 'Angkatan Kerja Pengangguran - Jumlah',
        'Angkatan Kerja - Jumlah Angkatan Kerja', 
        'Angkatan Kerja + Bukan Angkatan Kerja (Jumlah )'
    ]
    final_df = final_df[cols_to_keep].drop_duplicates(subset=['id'])

    # 9. Ekspor Hasil
    output_file = 'data/integrated_job_market_java_v2.csv'
    final_df.to_csv(output_file, index=False)
    
    print(f"\n--- RINGKASAN INTEGRASI ---")
    print(f"Total Lowongan Input: {len(df_js)}")
    print(f"Berhasil Dipetakan: {len(final_df)} ({len(final_df)/len(df_js)*100:.1f}%)")
    print(f"Dataset terintegrasi disimpan di: {output_file}")

if __name__ == "__main__":
    main()


## Tahap 5: Penghitungan Opportunity Index & Integrasi Data Spasial
Tahap ini menyatukan daftar 119 batas wilayah administratif GeoJSON Pulau Jawa dengan data BPS, koordinat, dan volume lowongan kerja.
Kami menghitung **Opportunity Index** untuk masing-masing wilayah dengan formula:
$$Opportunity\ Index = \frac{Total\ Lowongan}{Jumlah\ Angkatan\ Kerja\ (BPS)}$$
Ini menggambarkan seberapa besar peluang penyerapan kerja nyata per individu pencari kerja. Data wilayah yang tidak memiliki data lowongan diisi dengan volume 0 (mencegah data loss saat visualisasi di Streamlit), dan status kesejahteraan dikategorikan menjadi **Lautan Peluang** atau **Zona Merah** berdasarkan median indeks regional.

In [ ]:
import pandas as pd
import numpy as np
import os
import json

"""
TAHAP 4: PERHITUNGAN OPPORTUNITY INDEX & INTEGRASI DATA SPASIAL
Penulis: Antigravity AI
Deskripsi: Script ini menggabungkan batas administratif GeoJSON, data sosio-ekonomi BPS, 
           koordinat wilayah, dan volume pekerjaan dari Jobstreet secara utuh.
           TF-IDF dan pemrosesan teks NLP telah dihapus untuk berfokus pada Data Science Spasial.
"""

def get_qualification_score(title):
    title = str(title).lower()
    if any(k in title for k in ['manager', 'kepala', 'director', 'lead', 'senior', 'head', 'vp', 'chief']):
        return 3
    if any(k in title for k in ['specialist', 'supervisor', 'coordinator', 'analyst', 'spv', 'expert']):
        return 2
    return 1

def main():
    print("=== TAHAP 4: INTEGRASI DATA & OPPORTUNITY INDEX ===")
    
    data_dir = 'data'
    input_file = os.path.join(data_dir, 'integrated_job_market_java_v2.csv')
    geojson_file = os.path.join(data_dir, 'java_regencies.geojson')
    coord_file = os.path.join(data_dir, 'java_regency_coordinates.csv')
    bps_file = os.path.join(data_dir, 'master_bps_socioeconomic.csv')
    
    # Validasi file input
    for f_path in [input_file, geojson_file, coord_file, bps_file]:
        if not os.path.exists(f_path):
            print(f"Error: File '{f_path}' tidak ditemukan.")
            return
            
    df_jobs = pd.read_csv(input_file)
    
    # 1. LOAD MASTER LIST DARI GEOJSON (119 Wilayah)
    print("Memuat Master Wilayah dari GeoJSON...")
    with open(geojson_file, 'r') as f:
        g_data = json.load(f)
    master_names = sorted(list(set([f['properties']['clean_name'] for f in g_data['features']])))
    master_df = pd.DataFrame(master_names, columns=['matched_regency'])
    
    # Standardisasi nama untuk menyelaraskan GeoJSON dengan BPS & Koordinat
    # GeoJSON menggunakan: "Administrasi Kepulauan Seribu" dan "Gunungkidul"
    # BPS & Koordinat menggunakan: "Kepulauan Seribu" dan "Gunung Kidul"
    def std_name(name):
        if not isinstance(name, str): return ""
        name = name.strip()
        if name == 'Administrasi Kepulauan Seribu':
            return 'Kepulauan Seribu'
        if name == 'Gunungkidul':
            return 'Gunung Kidul'
        return name
        
    master_df['join_key'] = master_df['matched_regency'].apply(std_name)
    
    # 2. LOAD & INTEGRASIKAN DATA SOSIO-EKONOMI BPS
    print("Mengintegrasikan data sosio-ekonomi BPS...")
    df_bps = pd.read_csv(bps_file)
    df_bps['join_key'] = df_bps['Kabupaten/Kota'].apply(std_name)
    df_bps_unique = df_bps.drop_duplicates(subset=['join_key'])
    
    # Gabungkan data Angkatan Kerja (Labor Force)
    hub_stats = pd.merge(master_df, df_bps_unique[['join_key', 'Provinsi', 'Angkatan Kerja - Jumlah Angkatan Kerja']], on='join_key', how='left')
    hub_stats.rename(columns={'Angkatan Kerja - Jumlah Angkatan Kerja': 'labor_force_num'}, inplace=True)
    
    # 3. LOAD & INTEGRASIKAN KOORDINAT WILAYAH
    print("Mengintegrasikan koordinat geospasial...")
    df_coords = pd.read_csv(coord_file)
    df_coords['join_key'] = df_coords['City_Name'].apply(std_name)
    df_coords_unique = df_coords.drop_duplicates(subset=['join_key'])
    
    hub_stats = pd.merge(hub_stats, df_coords_unique[['join_key', 'Latitude', 'Longitude']], on='join_key', how='left')
    
    # Hapus join_key sementara
    hub_stats.drop(columns=['join_key'], inplace=True)
    
    # 4. PROSES & AGREGASI DATA VOLUME LOWONGAN
    print("Memproses volume pekerjaan & indeks kualifikasi...")
    df_jobs['qual_score'] = df_jobs['title'].apply(get_qualification_score)
    
    job_stats = df_jobs.groupby('matched_regency').agg({
        'id': 'count',
        'qual_score': 'mean'
    }).rename(columns={'id': 'job_volume', 'qual_score': 'competitive_index'})
    
    # Konversi indeks job_stats (dari nama koordinat/BPS) ke nama GeoJSON
    def rev_std_name(name):
        if name == 'Kepulauan Seribu':
            return 'Administrasi Kepulauan Seribu'
        if name == 'Gunung Kidul':
            return 'Gunungkidul'
        return name
        
    job_stats.index = job_stats.index.map(rev_std_name)
    job_stats.index.name = 'matched_regency'
    
    # 5. GABUNGKAN LOWONGAN KE DATAFRAME UTAMA
    print("Menggabungkan statistik lowongan ke master wilayah...")
    hub_stats = pd.merge(hub_stats, job_stats, on='matched_regency', how='left')
    
    # Mengisi default jika wilayah tidak memiliki lowongan kerja
    hub_stats['job_volume'] = hub_stats['job_volume'].fillna(0).astype(int)
    hub_stats['competitive_index'] = hub_stats['competitive_index'].fillna(1.0)
    
    # 6. HITUNG INDEKS PELUANG (OPPORTUNITY INDEX)
    print("Menghitung Opportunity Index...")
    # Gunakan .replace(0, np.nan) untuk menghindari pembagian dengan nol
    hub_stats['opportunity_index'] = hub_stats['job_volume'] / hub_stats['labor_force_num'].replace(0, np.nan)
    hub_stats['opportunity_index'] = hub_stats['opportunity_index'].fillna(0.0)
    
    # 7. KLASIFIKASI KESEJAHTERAAN & MEMBERSIHKAN NAN
    # Gunakan median dari wilayah yang memiliki lowongan kerja untuk pengelompokan
    mask_has_jobs = hub_stats['job_volume'] > 0
    med_opp = hub_stats[mask_has_jobs]['opportunity_index'].median() if any(mask_has_jobs) else 0.0
    hub_stats['prosperity_status'] = np.where(
        (hub_stats['opportunity_index'] >= med_opp) & (hub_stats['job_volume'] > 5), 
        "Lautan Peluang", 
        "Zona Merah"
    )
    
    # Pengisian data koordinat/provinsi sisa (jika ada yang terlewat)
    hub_stats['Latitude'] = hub_stats['Latitude'].fillna(0.0)
    hub_stats['Longitude'] = hub_stats['Longitude'].fillna(0.0)
    hub_stats['Provinsi'] = hub_stats['Provinsi'].fillna("Jawa")
    
    # EXPORT KE CSV
    output_path = os.path.join(data_dir, 'java_job_market_final_analysis.csv')
    hub_stats.to_csv(output_path, index=False)
    
    print(f"Sukses! Data final terintegrasi disimpan: {output_path} (Total Wilayah: {len(hub_stats)})")

if __name__ == "__main__":
    main()


## Tahap 6: Klastering Spasial Multidimensi dengan DBSCAN
Pada tahap akhir, algoritma **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** digunakan untuk mengelompokkan wilayah-wilayah ke dalam aglomerasi klaster ekonomi rill secara spasial.
Untuk memastikan fitur numerik volume lowongan tidak mendominasi fitur spasial (koordinat Latitude & Longitude) yang berskala kecil:
1. Kita melakukan transformasi logaritmik (`np.log1p`) pada volume lowongan.
2. Melakukan standardisasi spasial menggunakan `StandardScaler`.
3. Menjalankan DBSCAN dengan radius epsilon `eps=0.7` dan `min_samples=3` untuk mengelompokkan area metropolitan.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import os

"""
TAHAP 5: KLASTERING SPASIAL & MULTIDIMENSI (DBSCAN + StandardScaler + Log Volume)
Penulis: Antigravity AI (Falah's Thesis Assistant)
Deskripsi: Script final menggunakan StandardScaler dan log-transformation untuk menormalisasi 
           job volume serta Density-Based Spatial Clustering (DBSCAN) untuk mengidentifikasi 
           "Hub Ekonomi" secara geografis spasial yang akurat.
"""

def main():
    print("Memulai Tahap Akhir: Klastering Spasial (DBSCAN)...")
    
    # 1. Memuat Data Hasil Analisis Indexing
    input_file = os.path.join('data', 'java_job_market_final_analysis.csv')
    df = pd.read_csv(input_file)
    
    # 2. Persiapan Fitur Kombinasi (Spasial + Numerik Opsional)
    # Filter wilayah dengan koordinat valid
    valid_coords_mask = (df['Latitude'] != 0.0) & (df['Longitude'] != 0.0)
    df_valid = df[valid_coords_mask].copy()
    
    if len(df_valid) >= 3:
        # Menambahkan opportunity_index (kemampuan menyerap) ke dalam matriks spasial
        features = df_valid[['Latitude', 'Longitude', 'job_volume']].copy().values
        
        # Log-transformation pada job_volume agar skalanya stabil dan tidak mendominasi jarak spasial
        features[:, 2] = np.log1p(features[:, 2])
        
        # StandardScaler untuk menormalisasi koordinat dan log-volume secara proporsional
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(features)
        
        # 3. Eksekusi DBSCAN
        # eps=0.7 dan min_samples=3 menghasilkan pengelompokan aglomerasi metropolitan yang kokoh
        db = DBSCAN(eps=0.7, min_samples=3).fit(features_scaled)
        df_valid['cluster_id'] = db.labels_
        
        # Evaluasi Cluster menggunakan DBCV jika memungkinkan
        try:
            import hdbscan
            valid_labels = db.labels_[db.labels_ != -1]
            if len(set(valid_labels)) > 1:
                dbcv_score = hdbscan.validity.validity_index(features_scaled, db.labels_)
                print(f"[EVALUASI KLASTER] DBCV Score: {dbcv_score:.4f} (-1.0 s/d 1.0)")
        except Exception as e:
            print(f"Informasi evaluasi klaster (DBCV): {e}")
    else:
        df_valid['cluster_id'] = -1

    # Gabungkan kembali dengan data original (wilayah tanpa koordinat mendapat ID -1)
    df = pd.merge(df, df_valid[['matched_regency', 'cluster_id']], on='matched_regency', how='left')
    df['cluster_id'] = df['cluster_id'].fillna(-1).astype(int)
    
    # 4. Pelabelan Klaster (Hub Status)
    df['hub_type'] = np.where(df['cluster_id'] == -1, 'Isolated zone', 'Economic Hub')
    
    # 5. Ringkasan Hasil Klastering
    labels = df_valid['cluster_id'].values if 'cluster_id' in df_valid.columns else np.array([])
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    print(f"\n--- HASIL KLASTERING SPASIAL ---")
    print(f"Total Cluster Hub Ditemukan: {n_clusters}")
    print(f"Total Wilayah Outlier (Noise): {list(labels).count(-1)}")
    
    # 6. Analisis Karakteristik Per Hub
    clusters_summary = []
    for cid in set(df['cluster_id']):
        if cid == -1: continue
        
        cluster_data = df[df['cluster_id'] == cid]
        avg_opportunity = cluster_data['opportunity_index'].mean()
        total_jobs = cluster_data['job_volume'].sum()
        top_province = cluster_data['Provinsi'].mode()[0] if not cluster_data['Provinsi'].empty else "Jawa"
        
        # Penentuan Status "Lautan Peluang" per Klaster
        status = "Lautan Peluang" if avg_opportunity > df['opportunity_index'].median() else "Zona Merah"
        
        clusters_summary.append({
            "Cluster_ID": cid,
            "Hub_Region": top_province,
            "Total_Jobs": total_jobs,
            "Avg_Opportunity": round(avg_opportunity, 5),
            "Status": status,
            "Member_Count": len(cluster_data)
        })

    if clusters_summary:
        summary_df = pd.DataFrame(clusters_summary)
        print("\nDetail Ringkasan Hub Ekonomi:")
        print(summary_df.to_string(index=False))
    
    # 7. Ekspor Hasil Akhir
    output_file = os.path.join('data', 'java_job_market_hubs_final.csv')
    df.to_csv(output_file, index=False)
    print(f"\nData klaster lengkap disimpan di: {output_file}")

if __name__ == "__main__":
    main()


## Tahap 7: Visualisasi Hasil Eksploratif (Matplotlib & Seaborn)
Mari kita plot koordinat wilayah Pulau Jawa yang berwarna berdasarkan ID Klaster DBSCAN dan berukuran proporsional terhadap volume lowongan kerja.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data/java_job_market_hubs_final.csv')

plt.figure(figsize=(14, 7))
sns.set_style("whitegrid")

# Filter wilayah dengan koordinat valid untuk plotting
df_plot = df[(df['Latitude'] != 0.0) & (df['Longitude'] != 0.0)]

scatter = plt.scatter(
    df_plot['Longitude'], 
    df_plot['Latitude'], 
    c=df_plot['cluster_id'], 
    s=df_plot['job_volume'] * 0.2 + 20, # Scaling visualisasi titik
    cmap='viridis', 
    alpha=0.6, 
    edgecolors='black', 
    linewidth=0.5
)

# Labeling Hub Utama (menggunakan rata-rata koordinat tiap klaster)
for cid in set(df_plot['cluster_id']):
    if cid == -1: continue
    cluster_subset = df_plot[df_plot['cluster_id'] == cid]
    centroid_lon = cluster_subset['Longitude'].mean()
    centroid_lat = cluster_subset['Latitude'].mean()
    plt.text(
        centroid_lon, 
        centroid_lat, 
        f"Hub {cid}", 
        fontsize=11, 
        fontweight='bold',
        bbox=dict(facecolor='white', alpha=0.7, edgecolor='gray', boxstyle='round,pad=0.3')
    )

plt.title('Aglomerasi Geospasial Hub Ekonomi Pulau Jawa (DBSCAN)', fontsize=14, pad=15)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.colorbar(scatter, label='Cluster ID')
plt.show()